# Traded volume and VWAP

The answers to Section 3 of the sample questions, `documentation/tex/questions`, run.
The questions concern the volume, the traded value and the volume-weighted average price of
Section 1.1 of the lecture notes, the sweep cost they are compared with, and the trades of a
LOBSTER message file of Section 1.2.

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
Prices are integer counts of ticks, and a traded value is in tick-shares.

In [ ]:
import contextlib
import io
import timeit

import numpy as np
import pandas as pd
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")

## `question.vwapOfABatchAgainstTheBookItLeaves`: the second order meets the book the first left

In [ ]:
def run():
    book = AggregateBook.from_levels(
        {1000: 100, 999: 200}, {1002: 120, 1003: 180}
    )
    stream = [
        market_order(1.0, 150, BUY),
        limit_order(2.0, 100, 1004, BUY),
    ]
    value = volume = 0
    for message in stream:
        book.apply(message, record=False)
        value += book.last_traded_value
        volume += book.last_traded_size
    print(value / volume)


printed = outcome(run)
print(printed)
verify("question.vwapOfABatchAgainstTheBookItLeaves", "C", [
    printed == "1002.12",
    printed == "1002.2",
    printed == "1002.52",
    printed == "1002.6",
    printed == "1002.92",
])

Each distractor is the answer of a misreading, computed here: the second order against the
initial book, the limit order resting without trading, the mean of the two orders' own
averages, and the second order filled at its own limit price.

In [ ]:
first = (120 * 1002 + 30 * 1003) / 150
assert (150 * first + 100 * 1002) / 250 == 1002.12
assert first == 1002.2
assert (first + 1003) / 2 == 1002.6
assert (150 * first + 100 * 1004) / 250 == 1002.92

## `question.vwapOfTwoBuysAgainstOne`: what lies between two orders

In [ ]:
def run():
    def batch_vwap(stream):
        book = AggregateBook.from_levels(
            {1000: 100, 999: 200}, {1002: 120, 1003: 180}
        )
        value = volume = 0
        for message in stream:
            book.apply(message, record=False)
            value += book.last_traded_value
            volume += book.last_traded_size
        return value / volume

    one = [market_order(1.0, 200, BUY)]
    two = [market_order(1.0, 100, BUY), market_order(2.0, 100, BUY)]
    cancelled = [
        market_order(1.0, 100, BUY),
        withdrawal(1.5, 20, 1002, SELL),
        market_order(2.0, 100, BUY),
    ]
    refilled = [
        market_order(1.0, 100, BUY),
        limit_order(1.5, 50, 1002, SELL),
        market_order(2.0, 100, BUY),
    ]
    for stream in (one, two, cancelled, refilled):
        print(batch_vwap(stream))


printed = outcome(run).split("\n")
print(printed)
verify("question.vwapOfTwoBuysAgainstOne", "B", [
    printed == ["1002.4", "1002.4", "1002.4", "1002.4"],
    printed == ["1002.4", "1002.4", "1002.5", "1002.15"],
    printed == ["1002.4", "1002.0", "1002.0", "1002.0"],
    printed == ["1002.4", "1002.4", "1002.4", "1002.15"],
])

## `question.sweepCostIsTheVwapOfOneOrderLessTheMid`: the mid before the order, and after it

In [ ]:
from unito26.lob.messages import ReportedDepth, SweepSize


def run():
    book = AggregateBook.from_levels(
        {1000: 100, 999: 200}, {1002: 120, 1003: 180}
    )
    cost = book.sweep_cost(BUY, SweepSize(160), ReportedDepth(5))
    before = book.mid_price
    book.apply(market_order(1.0, 160, BUY), record=False)
    vwap = book.last_traded_value / book.last_traded_size
    print(cost, vwap - before, vwap - book.mid_price)


printed = outcome(run)
print(printed)
verify("question.sweepCostIsTheVwapOfOneOrderLessTheMid", "A", [
    printed == "1.25 1.25 0.75",
    printed == "0.25 1.25 0.75",
    printed == "1.25 1.25 1.25",
    printed == "1.25 0.75 0.75",
])

## `question.sweepCostWalksTheGrid`: the walk term over grid levels and over occupied ones

The stem describes `AggregateBook.sweep_cost` as a call of `_sweep_cost` on the occupied
levels of the side consumed; that description is checked first.

In [ ]:
from unito26.lob.orderbook import _sweep_cost

book = AggregateBook.from_levels({1000: 100}, {1002: 120, 1004: 180})
assert book.sweep_cost(BUY, SweepSize(160), ReportedDepth(5)) == _sweep_cost(
    book.occupied_levels(SELL, ReportedDepth(5)), book.mid_price, (SweepSize(160),), BUY
)[0]

In [ ]:
def run():
    def walk_cost(book, size):
        asks = sorted(book.asks.items())
        filled = walk = 0
        for i, (price, resting) in enumerate(asks, start=1):
            taken = min(size - filled, resting)
            filled += taken
            walk += taken * (i - 1)
        return book.spread / 2 + walk / size

    book = AggregateBook.from_levels({1000: 100}, {1002: 120, 1004: 180})
    print(book.sweep_cost(BUY, SweepSize(160), ReportedDepth(5)),
          walk_cost(book, 160))


printed = outcome(run)
print(printed)
verify("question.sweepCostWalksTheGrid", "D", [
    printed == "1.5 1.5",
    printed == "1.25 1.25",
    printed == "0.5 0.25",
    printed == "1.5 1.25",
])

## `question.vwapIsARatioOfSums`: a ratio of sums against a mean of ratios

In [ ]:
def run():
    trades = pd.DataFrame({
        "Volume": [300, 0, 100],
        "TradedValue": [300600, 0, 100000],
    })
    a = trades.TradedValue.sum() / trades.Volume.sum()
    b = (trades.TradedValue / trades.Volume).mean()
    print(a, b)


printed = outcome(run)
print(printed)
a, b = printed.split()
verify("question.vwapIsARatioOfSums", "E", [
    a == "1001.5" and b == "1001.5",
    a == "1001.5" and b == "nan",
    a == "1001.0" and b == "1001.5",
    a == "1001.5" and b != "nan" and abs(float(b) - 667.3) < 0.1,
    a == "1001.5" and b == "1001.0",
])

## `question.vwapWindowAtATiedTimestamp`: the left edge of the window, and a repeated time

In [ ]:
from unito26.lob.session import rolling_sum, window_start


def run():
    times = np.array([1.0, 2.0, 2.0, 3.0])
    volume = np.array([100.0, 100.0, 300.0, 100.0])
    value = np.array([100200.0, 100300.0, 300000.0, 100100.0])
    start = window_start(times, 1.0)
    vwap = rolling_sum(start, value) / rolling_sum(start, volume)
    print(vwap.tolist())


printed = outcome(run)
print(printed)
verify("question.vwapWindowAtATiedTimestamp", "B", [
    printed == "[1002.0, 1000.75, 1000.75, 1001.0]",
    printed == "[1002.0, 1003.0, 1000.75, 1001.0]",
    printed == "[1002.0, 1002.5, 1001.0, 1000.8]",
    printed == "[1002.0, 1001.0, 1001.0, 1000.8]",
])

The distractors are the windows $(t-w, t]$ read on the times rather than on the rows, the
window closed on the left, and both at once.

In [ ]:
times = np.array([1.0, 2.0, 2.0, 3.0])
volume = np.array([100.0, 100.0, 300.0, 100.0])
value = np.array([100200.0, 100300.0, 300000.0, 100100.0])
for left, right in (("right", "right"), ("left", "row"), ("left", "right")):
    first = np.searchsorted(times, times - 1.0, side=left)
    last = np.arange(4) + 1 if right == "row" else np.searchsorted(times, times, side="right")
    print(left, right, [float(value[i:j].sum() / volume[i:j].sum()) for i, j in zip(first, last)])

## `question.rollingVwapByCumulativeSums`: cumulative sums against a walk back over each window

A session of $n$ trades over one trading day of 23400 seconds, and a window of 1800 seconds,
so that a window holds about $n/13$ trades. Each implementation is timed at
$n = 1000, 2000, 4000$; the choices are read off the growth of each time over the ladder and
off their ratio at $n = 4000$.

In [ ]:
def rolling_vwap_a(times, volume, value, w):
    start = window_start(times, w)
    return rolling_sum(start, value) / rolling_sum(start, volume)

In [ ]:
def rolling_vwap_b(times, volume, value, w):
    out = []
    for i, t in enumerate(times):
        traded = worth = 0.0
        j = i
        while j >= 0 and times[j] > t - w:
            traded += volume[j]
            worth += value[j]
            j -= 1
        out.append(worth / traded)
    return out

In [ ]:
def seconds(function, arguments, number):
    return min(timeit.repeat(lambda: function(*arguments), number=number, repeat=3)) / number


rng = np.random.default_rng(3)
timings = {}
for n in (1_000, 2_000, 4_000):
    times = np.sort(rng.uniform(0.0, 23_400.0, n))
    volume = rng.integers(1, 10, n) * 100.0
    value = volume * rng.integers(995, 1005, n)
    arguments = (times, volume, value, 1_800.0)
    assert np.allclose(rolling_vwap_a(*arguments), rolling_vwap_b(*arguments))
    timings[n] = (seconds(rolling_vwap_a, arguments, 20), seconds(rolling_vwap_b, arguments, 1))
    print(f"n = {n:>5}:  A {timings[n][0] * 1e3:8.3f} ms   B {timings[n][1] * 1e3:8.1f} ms")

growth_a = timings[4_000][0] / timings[1_000][0]
growth_b = timings[4_000][1] / timings[1_000][1]
b_over_a = timings[4_000][1] / timings[4_000][0]
print(f"growth of A {growth_a:.1f}x, growth of B {growth_b:.1f}x, B over A at 4000 {b_over_a:.0f}x")
verify("question.rollingVwapByCumulativeSums", "A", [
    b_over_a > 5 and growth_a < 6 and growth_b > 10,
    b_over_a > 1 and growth_a < 6 and growth_b < 6,
    b_over_a < 1,
    b_over_a > 1 and growth_a > 10 and growth_b < 6,
])

## `question.vwapOfAnEmptyWindowIsNan`: a window that traded nothing

In [ ]:
def vwap(value, volume):
    return np.where(volume > 0, value / volume, np.nan)

vwap_a = vwap

In [ ]:
def vwap(value, volume):
    return np.divide(
        value, volume, out=np.full(len(volume), np.nan), where=volume > 0
    )

vwap_b = vwap

In [ ]:
def vwap(value, volume):
    return value / np.maximum(volume, 1.0)

vwap_c = vwap

In [ ]:
def run(vwap):
    value = np.array([100200.0, 0.0, 100300.0])
    volume = np.array([100.0, 0.0, 100.0])
    mid = np.array([1001.0, 1001.0, 1001.5])
    with np.errstate(invalid="raise"):
        print(np.nanmean(vwap(value, volume) - mid))


a, b, c = outcome(run, vwap_a), outcome(run, vwap_b), outcome(run, vwap_c)
print("A:", a, "   B:", b, "   C:", c)


def near(printed, number):
    try:
        return abs(float(printed) - number) < 1
    except ValueError:
        return False


verify("question.vwapOfAnEmptyWindowIsNan", "D", [
    a == b == c == "1.25",
    a == b == "1.25" and near(c, -333),
    a == "FloatingPointError" and b == c == "1.25",
    a == "FloatingPointError" and b == "1.25" and near(c, -333),
    a == b == "FloatingPointError" and near(c, -333),
])

## `question.frameSweepCostZeroTimesNan`: a padded level in a vectorised sweep

In [ ]:
from unito26.lob.session import _frame_sweep_cost

sweep_cost_a = _frame_sweep_cost

In [ ]:
def sweep_cost(prices, sizes, mid, size, direction):
    before = np.cumsum(sizes, axis=1) - sizes
    taken = np.clip(size - before, 0.0, sizes)
    filled = taken.sum(1) >= size
    value = (taken * prices).sum(1)
    cost = direction * (value / size - mid)
    return np.where(filled, cost, np.nan), filled

sweep_cost_b = sweep_cost

In [ ]:
def run(sweep_cost):
    prices = np.array([
        [1002.0, 1003.0, 1004.0],
        [1002.0, 1003.0, np.nan],
        [1002.0, np.nan, np.nan],
    ])
    sizes = np.array([
        [120.0, 180.0, 50.0],
        [120.0, 180.0, 0.0],
        [120.0, 0.0, 0.0],
    ])
    mid = np.full(3, 1001.0)
    cost, filled = sweep_cost(prices, sizes, mid, 160, BUY)
    print(cost.tolist())


a, b = outcome(run, sweep_cost_a), outcome(run, sweep_cost_b)
print("A:", a, "   B:", b)
verify("question.frameSweepCostZeroTimesNan", "C", [
    a == b == "[1.25, 1.25, nan]",
    a == "[1.25, 1.25, 1.0]" and b == "[1.25, nan, 1.0]",
    a == "[1.25, 1.25, nan]" and b == "[1.25, nan, nan]",
    a == b == "[1.25, nan, nan]",
])

## `question.lobsterAggressorIsMinusDirection`: the direction of an execution names the passive side

In [ ]:
def run(alternative):
    messages = pd.DataFrame({
        "Type": [1, 4, 4, 3, 4, 4],
        "Size": [50, 96, 60, 50, 32, 20],
        "Price": [1001, 1002, 1000, 1001, 1003, 999],
        "Direction": [1, -1, 1, 1, -1, 1],
    })
    if alternative == "A":
        aggressor = -messages.Direction
    else:
        aggressor = messages.Direction
    executed = messages.Type == 4
    size = messages.Size.where(executed, 0)
    value = size * messages.Price
    buy = aggressor > 0
    print(value[buy].sum() / size[buy].sum(),
          value[~buy].sum() / size[~buy].sum())


a, b = outcome(run, "A"), outcome(run, "B")
print("A:", a, "   B:", b)
verify("question.lobsterAggressorIsMinusDirection", "B", [
    a == b == "1002.25 999.75",
    a == "1002.25 999.75" and b == "999.75 1002.25",
    a == "999.75 1002.25" and b == "1002.25 999.75",
    a == b == "999.75 1002.25",
])